# 07 · Песочница

Свои запросы к сохранённым адаптерам. Адаптеры методов на парах обучены поверх SFT, поэтому
сначала вливаем SFT; «база» здесь — SFT-модель. Метрики не считаются, в `runs/` ничего не пишется.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src import data, infer, metrics, report

import contextlib

import torch
from peft import PeftModel
from transformers import TextStreamer

model, tokenizer = infer.load_model()
if (report.RUNS / "sft-adapter").exists():
    model = PeftModel.from_pretrained(model, report.RUNS / "sft-adapter").merge_and_unload()

adapters = sorted(p.name.removesuffix("-adapter") for p in report.RUNS.glob("*-adapter") if p.name != "sft-adapter")
if adapters:
    model = PeftModel.from_pretrained(model, report.RUNS / f"{adapters[0]}-adapter", adapter_name=adapters[0])
    for name in adapters[1:]:
        model.load_adapter(report.RUNS / f"{name}-adapter", adapter_name=name)
print("адаптеры поверх SFT:", adapters or "нет")

In [ ]:
def ask(request, document="", dialog=None, adapter=None, max_new_tokens=600):
    """One answer for a hand-typed situation; adapter=None is the SFT model."""
    prompt = data.prompt_for(request, document, dialog)
    if adapter is None or not adapters:
        with (model.disable_adapter() if adapters else contextlib.nullcontext()):
            return infer.generate(model, tokenizer, [prompt], max_new_tokens=max_new_tokens)[0]
    model.set_adapter(adapter)
    return infer.generate(model, tokenizer, [prompt], max_new_tokens=max_new_tokens)[0]


def compare(request, document="", dialog=None):
    print("ЗАПРОС:", request)
    for name in [None] + adapters:
        print("═" * 78, name or "sft")
        print(ask(request, document, dialog, adapter=name))


compare("Сформулируй мне гипотезу, я пишу про выгорание медсестёр", document=data.DOCUMENTS["ext-nurse-results"])

In [ ]:
compare("Да он придирается, гипотеза нормальная",
        document=data.DOCUMENTS["ext-music-intro"],
        dialog=[{"user": "Руководитель говорит, что гипотеза неконкретная. Что делать?",
                 "assistant": "Он про фразу «дети будут лучше учиться»: из неё не видно, что измерять. Каким показателем сравниваете группы?"}])

In [ ]:
rows = {r["id"]: r for split in ("test_product", "test_extended", "dev") for r in data.load(split)}


def inspect(row_id, adapter=None):
    """Reference answer, the model's answer, its checks and the judge's verdict."""
    row = rows[row_id]
    data.show(row)
    if adapter and adapters:
        model.set_adapter(adapter)
    answer = infer.generate(model, tokenizer, [row["prompt"]])[0]
    print(f"ОТВЕТ [{adapter or 'sft'}]:\n{answer}\n")
    print("проверки:", metrics.run(answer, data.case(row)))
    print("судья:", "PASS" if infer.judge(model, tokenizer, [row], [answer])[0] else "FAIL")


inspect("EXT-RME-01", adapter=adapters[0] if adapters else None)

In [ ]:
def stream(request, document="", adapter=None, max_new_tokens=600):
    """Token by token, for long answers."""
    if adapter and adapters:
        model.set_adapter(adapter)
    batch = tokenizer.apply_chat_template([data.prompt_for(request, document)], add_generation_prompt=True, enable_thinking=False,
                                          tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id,
                       streamer=TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True))


stream("Сократи введение до одного абзаца", document=data.DOCUMENTS["ext-phil-intro"], adapter=adapters[0] if adapters else None)